# PyTorch 入门第二课：搭建神经网络的积木 —— `nn.Module`

在深度学习中，无论你是手写一个简单的分类器，还是调用千亿参数的大语言模型（如 GPT、Qwen），其底层都是由一个个模块拼接而成的。

PyTorch 提供了 `torch.nn.Module` 这个基类。**所有的神经网络模型、甚至模型内部的一层（Layer），都必须继承自这个类。**

本节课我们将深入学习：
1. 如何使用最基础的神经网络算子（`Linear`, `ReLU`, `LayerNorm`）。
2. 如何继承 `nn.Module` 搭建一个经典的 **3 层多层感知机（MLP）**。
3. **科研中的工程化应用总结**（模型模式切换、权重冻结与参数遍历）。

## 1. 认识核心算子（核心层）

在 `torch.nn` 模块下，有大量预定义好的层。我们先来看看最常用的三个：

In [1]:
import torch
import torch.nn as nn

# 1. 线性层 (全连接层) nn.Linear
# 它的数学本质是：y = xA^T + b
# in_features 是输入特征的维度，out_features 是输出特征的维度。
linear_layer = nn.Linear(in_features=128, out_features=64)
x = torch.randn(32, 128) # 模拟一个 Batch=32, 特征为128 的输入数据
out_linear = linear_layer(x)
print("Linear 层输出形状:", out_linear.shape) # 结果应为 [32, 64]

# 2. 激活函数 nn.ReLU 与 nn.GELU
# 激活函数负责给神经网络注入“非线性”能力。大模型中如今更偏爱 GELU（更平滑）。
relu = nn.ReLU()
gelu = nn.GELU()
out_gelu = gelu(out_linear)

# 3. 层归一化 nn.LayerNorm
# 在 Transformer 架构（包括大模型）中，极其高频地使用 LayerNorm 来稳定数据分布。
layer_norm = nn.LayerNorm(normalized_shape=64)
out_norm = layer_norm(out_gelu)
print("LayerNorm 后的输出形状仍然是:", out_norm.shape) # 不改变形状，只改变数据的分布

Linear 层输出形状: torch.Size([32, 64])
LayerNorm 后的输出形状仍然是: torch.Size([32, 64])


## 2. 核心实战：搭建一个 3 层 MLP

编写自定义模型**永远遵循两大步**：
1. **`__init__` 函数**：初始化。把你想用的组件（积木）先买回来放在这里。
2. **`forward` 函数**：前向传播。定义数据如何穿过这些组件（数据流向）。

接下来我们手撸一个 3 层 MLP：

In [2]:
class ThreeLayerMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        # 1. 必须先调用父类的初始化方法，这一步绝不能省
        super(ThreeLayerMLP, self).__init__()
        
        # 2. 声明网络层（如同买积木）
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.act1 = nn.GELU()
        
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.act2 = nn.GELU()
        
        # 最后一层通常不需要激活函数，直接输出预测的分数（Logits）
        self.layer3 = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # 3. 拼接积木：规定数据 x 是怎么一层层往下流动的
        x = self.layer1(x)
        x = self.norm1(x)
        x = self.act1(x)
        
        x = self.layer2(x)
        x = self.norm2(x)
        x = self.act2(x)
        
        logits = self.layer3(x)
        return logits

# 实例化这个网络，输入维度 256，隐藏层 512，输出类别为 10（比如 10 分类任务）
model = ThreeLayerMLP(input_dim=256, hidden_dim=512, output_dim=10)
print(model)

# 模拟一条数据通过网络
dummy_input = torch.randn(16, 256) # Batch size=16, feature_dim=256
output = model(dummy_input)
print("\n最终网络输出形状:", output.shape) # 期望是 [16, 10]

ThreeLayerMLP(
  (layer1): Linear(in_features=256, out_features=512, bias=True)
  (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
  (act1): GELU(approximate='none')
  (layer2): Linear(in_features=512, out_features=512, bias=True)
  (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
  (act2): GELU(approximate='none')
  (layer3): Linear(in_features=512, out_features=10, bias=True)
)

最终网络输出形状: torch.Size([16, 10])


## 3. 科研与工程化应用：避坑与高阶技巧

在真实的科研实验或生产工程中，掌握 `nn.Module` 的以下高级操作至关重要：

### 3.1 训练模式与推理模式切换（踩坑重灾区）
某些层（例如 `nn.Dropout` 和 `nn.BatchNorm`）在**训练**和**推理**时的行为是完全不同的。
- 科研新手常犯的错误：测试模型时忘记关掉 Dropout，导致每次评估的准确率在波动。
- **规范做法**：
  - 训练前调用 `model.train()`
  - 评估/测试前**一定**要调用 `model.eval()`

In [ ]:
# 在科研代码主循环中：
model.train() # 开启训练模式（Dropout等正常工作）
# --- 执行训练逻辑 ---

with torch.no_grad(): # 推理时不计算梯度，节省显存
    model.eval()  # 开启评估模式（关闭Dropout等，保证预测结果的确定性）
    # --- 执行评估/验证逻辑 ---

### 3.2 参数遍历与大模型微调（冻结部分权重）
在微调大语言模型（如 PEFT / LoRA 方法）时，我们通常只希望训练部分参数，而冻结底层参数。通过 `model.named_parameters()`，你可以非常方便地遍历并控制每一个参数的状态。

In [ ]:
print("遍历网络中的参数名与状态：")
for name, param in model.named_parameters():
    # 假设我们只想训练最后一层（layer3），冻结前面的所有层
    if "layer3" not in name:
        param.requires_grad = False # 冻结此参数，不计算梯度，不更新权重
    
    print(f"参数名: {name: <15} | 形状: {str(list(param.shape)): <15} | 是否参与训练 (requires_grad): {param.requires_grad}")

### 3.3 科研模块化设计思维
在上面的 `ThreeLayerMLP` 中，我们将 `Linear -> LayerNorm -> GELU` 重复写了两遍。在大型科研项目中，这会显得极其臃肿。

**工程化思维**：将重复的逻辑抽离成一个子 `Module`，或者使用 `nn.Sequential`。

In [ ]:
# 使用 nn.Sequential 大幅精简代码结构 (适合流水线式、没有复杂分支的网络)
class ElegantMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        
        self.block1 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )
        
        self.block2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )
        
        self.classifier = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        return self.classifier(x)

elegant_model = ElegantMLP(256, 512, 10)
print(elegant_model)